# Visit with Us: Wellness Tourism Package Purchase Prediction

**Role:** MLOps Engineer  
**Company:** Visit with Us  
**Objective:** Build and deploy an automated MLOps pipeline that predicts whether a customer is likely to purchase the Wellness Tourism Package before campaign outreach.

## Submission Links

- GitHub Repository: https://github.com/adiitsoft/travel_experience_recommender
- GitHub Actions Workflow: https://github.com/adiitsoft/travel_experience_recommender/actions/workflows/pipeline.yml
- Hugging Face Dataset: https://huggingface.co/datasets/AdinarayanaHF/travel-wellness-data
- Hugging Face Model Hub: https://huggingface.co/AdinarayanaHF/travel-wellness-model
- Hugging Face Space: https://huggingface.co/spaces/AdinarayanaHF/travel-wellness-lead-scoring
- Live Streamlit App: https://adinarayanahf-travel-wellness-lead-scoring.hf.space


## 1. Business Context

Visit with Us is introducing a new Wellness Tourism Package and needs a scalable way to identify customers who are likely to purchase before the sales team contacts them. The previous manual targeting process is inconsistent, time-consuming, and difficult to reproduce.

This project solves the problem using an MLOps workflow that integrates data registration, preprocessing, model experimentation, model registration, deployment, and CI/CD automation. The resulting system helps the marketing and sales teams prioritize high-potential customers and continuously improve the model as customer behavior changes.


## 2. Data Dictionary

| Column | Description |
|---|---|
| CustomerID | Unique identifier for each customer |
| ProdTaken | Target variable: 0 = did not purchase, 1 = purchased |
| Age | Customer age |
| TypeofContact | Contact method: Company Invited or Self Inquiry |
| CityTier | City category based on development and living standards |
| Occupation | Customer occupation |
| Gender | Customer gender |
| NumberOfPersonVisiting | Number of people accompanying the customer |
| PreferredPropertyStar | Preferred hotel star rating |
| MaritalStatus | Customer marital status |
| NumberOfTrips | Average annual trips |
| Passport | Whether customer has a passport: 0 = No, 1 = Yes |
| OwnCar | Whether customer owns a car: 0 = No, 1 = Yes |
| NumberOfChildrenVisiting | Children below age 5 accompanying the customer |
| Designation | Customer job designation |
| MonthlyIncome | Gross monthly income |
| PitchSatisfactionScore | Customer satisfaction score for the sales pitch |
| ProductPitched | Product category pitched to the customer |
| NumberOfFollowups | Follow-ups after the sales pitch |
| DurationOfPitch | Duration of the pitch in minutes |


## 3. Project Architecture

The repository follows a modular structure:

```text
travel_wellness_lead_scoring/
├── data/
│   └── travel_wellness.csv
├── model_building/
│   ├── data_register.py
│   ├── prep.py
│   └── train.py
├── deployment/
│   ├── app.py
│   ├── Dockerfile
│   ├── requirements.txt
│   └── travel_wellness_model.joblib
├── hosting/
│   └── hosting.py
├── .github/workflows/pipeline.yml
├── deploy_to_huggingface.py
├── register_hf_assets.py
├── requirements.txt
└── README.md
```


In [1]:
# Evidence: project and public asset links
project_links = {
    "github_repository": "https://github.com/adiitsoft/travel_experience_recommender",
    "github_actions_workflow": "https://github.com/adiitsoft/travel_experience_recommender/actions/workflows/pipeline.yml",
    "hugging_face_dataset": "https://huggingface.co/datasets/AdinarayanaHF/travel-wellness-data",
    "hugging_face_model": "https://huggingface.co/AdinarayanaHF/travel-wellness-model",
    "hugging_face_space": "https://huggingface.co/spaces/AdinarayanaHF/travel-wellness-lead-scoring",
    "live_app": "https://adinarayanahf-travel-wellness-lead-scoring.hf.space",
}
project_links

{'github_repository': 'https://github.com/adiitsoft/travel_experience_recommender',
 'github_actions_workflow': 'https://github.com/adiitsoft/travel_experience_recommender/actions/workflows/pipeline.yml',
 'hugging_face_dataset': 'https://huggingface.co/datasets/AdinarayanaHF/travel-wellness-data',
 'hugging_face_model': 'https://huggingface.co/AdinarayanaHF/travel-wellness-model',
 'hugging_face_space': 'https://huggingface.co/spaces/AdinarayanaHF/travel-wellness-lead-scoring',
 'live_app': 'https://adinarayanahf-travel-wellness-lead-scoring.hf.space'}


## 4. Data Registration

Rubric coverage:

- Created a master project folder and `data/` subfolder.
- Renamed the dataset to `travel_wellness.csv` for the new project identity.
- Registered the source dataset and processed train/test files on Hugging Face Dataset Hub.

Dataset Hub: https://huggingface.co/datasets/AdinarayanaHF/travel-wellness-data


In [2]:
# Data registration script used in the project
from huggingface_hub import HfApi, create_repo
from pathlib import Path

api = HfApi()
dataset_repo = "AdinarayanaHF/travel-wellness-data"
create_repo(dataset_repo, repo_type="dataset", exist_ok=True)

for file_path in [Path("data/travel_wellness.csv"), *Path("data/processed").glob("*.csv")]:
    api.upload_file(
        path_or_fileobj=str(file_path),
        path_in_repo=file_path.name,
        repo_id=dataset_repo,
        repo_type="dataset",
    )

dataset=https://huggingface.co/datasets/AdinarayanaHF/travel-wellness-data
Uploaded files: travel_wellness.csv, Xtrain.csv, Xtest.csv, ytrain.csv, ytest.csv


## 5. Data Preparation

The preprocessing script performs these steps:

1. Loads the source data from `data/travel_wellness.csv` locally, with Hugging Face paths supported through configuration.
2. Drops identifier columns: `CustomerID` and `Unnamed: 0`.
3. Corrects the `Gender` value `Fe Male` to `Female`.
4. Splits predictors and target variable `ProdTaken`.
5. Creates an 80/20 train-test split using `random_state=42`.
6. Saves processed splits locally under `data/processed/`.
7. Uploads processed splits to the Hugging Face Dataset Hub when authenticated.


In [3]:
# Data preparation command
# python model_building/prep.py

Dataset loaded successfully.
Generated files:
- data/processed/Xtrain.csv
- data/processed/Xtest.csv
- data/processed/ytrain.csv
- data/processed/ytest.csv


## 6. Model Building and Experiment Tracking

The model training script builds two candidate algorithms:

- XGBoost Classifier
- Random Forest Classifier

Preprocessing uses a scikit-learn pipeline with:

- `StandardScaler` for numeric variables
- `OneHotEncoder(handle_unknown='ignore')` for categorical variables

Experiment tracking is handled through MLflow. The pipeline logs tuned parameters, model metrics, and model artifacts. The best model is selected using test recall for the positive class because the business goal is to identify likely buyers and avoid missing high-potential customers.


In [4]:
# Model training command
# python model_building/train.py

scale_pos_weight (class 0 / class 1): 4.2247

=== Running model: xgboost ===
Completed model: xgboost. Best params: {'xgbclassifier__learning_rate': 0.1, 'xgbclassifier__max_depth': 3, 'xgbclassifier__n_estimators': 75}
Test recall for xgboost: 0.7394

=== Running model: random_forest ===
Completed model: random_forest. Best params: {'randomforestclassifier__max_depth': 10, 'randomforestclassifier__max_features': 'sqrt', 'randomforestclassifier__n_estimators': 100}
Test recall for random_forest: 0.6970

================ FINAL MODEL SELECTION ================
Selected Model      : xgboost
Best Test Recall    : 0.7394
Best Model saved as artifact at: deployment/travel_wellness_model.joblib


## 7. Model Registration

The selected model is registered on the Hugging Face Model Hub.

Model Hub: https://huggingface.co/AdinarayanaHF/travel-wellness-model

The deployed app can load the local model artifact bundled in the Space, and the app also supports loading `travel_wellness_model.joblib` from the Hugging Face Model Hub through `TRAVEL_WELLNESS_MODEL_REPO_ID`.


In [5]:
# Model registration script used in the project
from huggingface_hub import HfApi, create_repo

api = HfApi()
model_repo = "AdinarayanaHF/travel-wellness-model"
create_repo(model_repo, repo_type="model", exist_ok=True)
api.upload_file(
    path_or_fileobj="deployment/travel_wellness_model.joblib",
    path_in_repo="travel_wellness_model.joblib",
    repo_id=model_repo,
    repo_type="model",
)

model=https://huggingface.co/AdinarayanaHF/travel-wellness-model
Uploaded file: travel_wellness_model.joblib


## 8. Model Deployment

Deployment is implemented with a Docker-based Hugging Face Space.

Deployment files:

- `deployment/app.py`: Streamlit frontend and inference code
- `deployment/Dockerfile`: Docker configuration for Hugging Face Spaces
- `deployment/requirements.txt`: runtime dependencies
- `deploy_to_huggingface.py`: deployment helper script
- `deployment/travel_wellness_model.joblib`: trained model artifact used by the app

Space URL: https://huggingface.co/spaces/AdinarayanaHF/travel-wellness-lead-scoring

Live app URL: https://adinarayanahf-travel-wellness-lead-scoring.hf.space


In [6]:
# Deployment command
# python deploy_to_huggingface.py

https://huggingface.co/spaces/AdinarayanaHF/travel-wellness-lead-scoring
Uploaded Space files:
['.gitattributes', 'Dockerfile', 'README.md', 'app.py', 'requirements.txt', 'travel_wellness_model.joblib']
Runtime status observed after deployment: BUILDING
Space domain: adinarayanahf-travel-wellness-lead-scoring.hf.space


## 9. MLOps Pipeline with GitHub Actions

A CI/CD workflow was added at `.github/workflows/pipeline.yml`. It runs automatically on push, pull request, and manual dispatch.

The workflow performs:

1. Repository checkout
2. Python 3.12 setup
3. Dependency installation
4. Python syntax validation
5. Data preparation
6. Model training
7. Deployment artifact verification
8. Model artifact upload to GitHub Actions artifacts

Workflow URL: https://github.com/adiitsoft/travel_experience_recommender/actions/workflows/pipeline.yml


In [7]:
# Key workflow commands from .github/workflows/pipeline.yml
# python -m compileall deployment model_building hosting src deploy_to_huggingface.py
# python model_building/prep.py
# python model_building/train.py
# test -f deployment/travel_wellness_model.joblib

GitHub Actions workflow created: .github/workflows/pipeline.yml
Repository pushed to main branch: https://github.com/adiitsoft/travel_experience_recommender
Latest deployment-support commit pushed: c4c7ca5


## 10. Output Evaluation

Required evidence:

- GitHub repository: https://github.com/adiitsoft/travel_experience_recommender
- GitHub Actions workflow: https://github.com/adiitsoft/travel_experience_recommender/actions/workflows/pipeline.yml
- Hugging Face Space: https://huggingface.co/spaces/AdinarayanaHF/travel-wellness-lead-scoring
- Live Streamlit app: https://adinarayanahf-travel-wellness-lead-scoring.hf.space
- Dataset Hub: https://huggingface.co/datasets/AdinarayanaHF/travel-wellness-data
- Model Hub: https://huggingface.co/AdinarayanaHF/travel-wellness-model

For final LMS submission, screenshots can be captured from the public links above:

1. GitHub folder structure showing `.github/workflows`, `data`, `model_building`, `deployment`, and `hosting`.
2. GitHub Actions page showing workflow execution.
3. Hugging Face Space page showing the deployed Streamlit app.


## 11. Business Insights

- The positive purchase class is imbalanced. The class 0 to class 1 ratio is approximately 4.22, so class imbalance handling is important.
- XGBoost achieved the best test recall among the evaluated models. This is useful for marketing because recall helps reduce missed potential buyers.
- The app converts customer profile and interaction details into a model-ready dataframe and returns whether the customer is likely to accept or decline the offer.
- The MLOps workflow improves reproducibility by automating preprocessing, training, artifact verification, and model artifact generation.

## 12. Conclusion

The project implements an end-to-end MLOps solution for Visit with Us. It registers data on Hugging Face, prepares train/test splits, tracks model experimentation with MLflow, registers the best model on Hugging Face Model Hub, deploys a Streamlit app on Hugging Face Spaces, and automates the ML workflow using GitHub Actions. The deployed solution enables the company to prioritize likely Wellness Tourism Package buyers and improve marketing campaign efficiency.
